# Pilot independent-review and scale gate

This notebook turns the feasibility pilot into an explicit collection decision. It does not perform or imitate the independent human review. Until all 13 targets are reviewed and discussion items are reconciled, the correct overall status is `blocked_pending_review`.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

from f1stewards.config import load_analysis_thresholds
from f1stewards.readiness import (
    evaluate_pilot_readiness,
    load_pilot_manual_records,
    readiness_decision,
)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DB_PATH = ROOT / 'data' / 'processed' / 'f1_stewarding.duckdb'
CODED_PATH = ROOT / 'data' / 'manual' / 'pilot_coded_adjudications.csv'
IMPACT_PATH = ROOT / 'data' / 'manual' / 'pilot_impact_assessments.csv'
REVIEW_PATH = ROOT / 'data' / 'manual' / 'pilot_independent_review.csv'

coded, impacts, reviews = load_pilot_manual_records(CODED_PATH, IMPACT_PATH, REVIEW_PATH)
review_frame = pd.read_csv(REVIEW_PATH, keep_default_na=False)

## 1. Independent-review status

The first-pass values remain in separate files. This packet records agreement, corrections, unresolved discussion, and measured review minutes without overwriting the original coding.

In [2]:
review_summary = (
    review_frame.groupby(['target_type', 'review_status'], dropna=False)
    .size()
    .rename('targets')
    .reset_index()
)
display(review_summary)
display(
    review_frame.loc[
        review_frame.review_status.eq('pending'),
        ['review_id', 'target_type', 'initial_summary', 'evidence_urls'],
    ]
)

,target_type,review_status,targets
0,adjudication,pending,9
1,impact_assessment,pending,4


,review_id,target_type,initial_summary,evidence_urls
0,review-adj-2019-aut-r69-t3-16,adjudication,Car 16 / lap 69 / turn 3 / collision / no acti...,https://www.fia.com/sites/default/files/decisi...
1,review-adj-2019-aut-r69-t3-33,adjudication,Car 33 / lap 69 / turn 3 / collision / no acti...,https://www.fia.com/sites/default/files/decisi...
2,review-adj-2023-abu-r15-t6-44,adjudication,Car 44 / lap 15 / turn 6 / collision / no acti...,https://www.fia.com/sites/default/files/decisi...
3,review-adj-2023-abu-r47-t6-11,adjudication,Car 11 / lap 47 / turn 6 / collision / five se...,https://www.fia.com/sites/default/files/decisi...
4,review-adj-2025-aut-r31-t4-22,adjudication,Car 22 / global lap 31 / turn 4 / collision / ...,https://www.fia.com/system/files/decision-docu...
5,review-adj-2025-aut-r4-t3-43,adjudication,Car 43 / lap 4 / turn 3 / alleged forcing off ...,https://www.fia.com/system/files/decision-docu...
6,review-adj-2025-aut-r54-t34-43,adjudication,Car 43 / global lap 54 / turn unresolved / for...,https://www.fia.com/system/files/decision-docu...
7,review-adj-2025-aut-r1-t3-12,adjudication,Car 12 / lap 1 / turn 3 / collision / three gr...,https://www.fia.com/system/files/decision-docu...
8,review-adj-2025-aut-r15-t3-22,adjudication,Car 22 / lap 15 / turn 3 / alleged forcing off...,https://www.fia.com/system/files/decision-docu...
9,review-impact-2023-abu-11,impact_assessment,Mechanical removal / P4 to P2 / two positions ...,https://www.fia.com/sites/default/files/decisi...


## 2. Measured scale gates

Objective gates are computed from the warehouse and validated manual files. `conditional_pass` preserves a bounded source limitation; `requires_decision` marks a judgment that code should not make.

In [3]:
with duckdb.connect(str(DB_PATH), read_only=True) as connection:
    gates = evaluate_pilot_readiness(
        connection, coded, impacts, reviews, load_analysis_thresholds()
    )

overall = readiness_decision(gates)
display(gates)
print(f'Overall: {overall}')

snapshot_path = ROOT / 'reports' / 'generated' / 'pilot_scale_readiness.csv'
snapshot_path.parent.mkdir(parents=True, exist_ok=True)
gates.to_csv(snapshot_path, index=False)
print(f'Wrote {snapshot_path}')

,gate,status,metric,note
0,discovery_and_retrieval,pass,156 source records; 2 recalled represented; 0 ...,Archive-visible recalls remain in lineage even...
1,decision_parsing,pass,25/26 Decision sections; 25/26 complete core (...,The incomplete record remains visible for manu...
2,source_lineage,conditional_pass,2 recalled records retained; version gaps regi...,Conditional while an event-date regulatory bin...
3,event_date_law_and_guidance,conditional_pass,3/3 Sporting Regulation selections verified; 3...,A later binary is never substituted for an unr...
4,timing_and_classification,pass,3/3 FastF1 events; 3/3 official final classifi...,Classification arithmetic is separately valida...
5,coding_validity,pass,9 adjudications / 8 incidents; 4 impact record...,All rows passed controlled-field and impossibl...
6,independent_review,pending,0/13 complete; 0 agree; 0 correct; 0 discussio...,All targets and discussion items must be resol...
7,review_burden,pending,Measured after all independent reviews are com...,A person must decide whether measured minutes ...
8,analytical_yield,requires_decision,8 candidate incidents across 3 pilot events (2...,The purposive three-event pilot is not a power...


Overall: blocked_pending_review
Wrote C:\Users\Brian Zeng\Documents\Codex\2026-08-08\https-www-usajobs-gov-job-876261000\work\f1-stewarding-analysis\reports\generated\pilot_scale_readiness.csv


## 3. Decision rule

A technical pass does not authorize full collection. While review is incomplete, stop at the pilot. Once every target is complete, use measured burden, disagreement patterns, source gaps, and expected analytical yield to choose `go`, `go with reduced scope`, or `stop/revise` under `docs/scale_decision_framework.md`. The software deliberately ends at `human_go_no_go_required`.